# PRAMAAN — Sovereign LLM on Google Colab via vLLM + ngrok

Run an open-weights LLM (Qwen 2.5 7B Instruct by default) on Colab's free T4 GPU and expose an OpenAI-compatible endpoint to your local PRAMAAN backend.

**Use this when:**
- you don't have a paid OpenAI / Anthropic / Groq key,
- you want the demo to honor the sovereignty pillar (no third-party hosted API),
- your laptop doesn't have a GPU.

## How it works

1. Colab spins up a T4 (free tier) or L4/A100 (Pro).
2. We install vLLM and load `Qwen/Qwen2.5-7B-Instruct`.
3. vLLM serves an OpenAI-compatible endpoint on `:8000`.
4. ngrok tunnels that out to a public HTTPS URL.
5. Set the URL in your local `.env` as `PRAMAAN_LLM_BASE_URL` and you're done.

> Note: Qwen 2.5 7B is the largest that comfortably fits on a free T4 (16 GB).
> If you have Colab Pro with an A100, you can swap to `Qwen/Qwen2.5-14B-Instruct`
> or `meta-llama/Llama-3.1-8B-Instruct`.

## 0 · Confirm the GPU

In [ ]:
!nvidia-smi

## 1 · Install vLLM and pyngrok

First-time install takes ~3-5 minutes. Subsequent runs reuse the wheel.

In [ ]:
%pip install -q --upgrade pip
%pip install -q vllm==0.6.4 pyngrok

## 2 · Set your ngrok auth token

Get a free token at https://dashboard.ngrok.com/get-started/your-authtoken (no card needed).

In [ ]:
import getpass, os
os.environ['NGROK_AUTHTOKEN'] = getpass.getpass('Paste your ngrok authtoken: ')

## 3 · Pick the model

Qwen 2.5 7B Instruct is the default — small enough for free Colab, strong on document extraction in English and Hindi.

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'  # alternatives: meta-llama/Llama-3.1-8B-Instruct
PORT = 8000

## 4 · Boot vLLM

First run will download the model weights (~16 GB) to Colab's disk. Cell streams logs; wait until you see `Uvicorn running on http://0.0.0.0:8000`.

In [ ]:
import subprocess, threading, time

vllm_cmd = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL_ID,
    '--port', str(PORT),
    '--max-model-len', '8192',
    '--gpu-memory-utilization', '0.90',
    '--dtype', 'auto',
    '--seed', '42',
]

proc = subprocess.Popen(vllm_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

def _stream(p):
    for line in p.stdout:
        print(line, end='')

t = threading.Thread(target=_stream, args=(proc,), daemon=True)
t.start()

# Heuristic wait: 4-6 minutes on free T4 to fully load 7B Qwen.
print('Waiting for vLLM to come up...')
time.sleep(60)
print('(Continuing while the server keeps loading; tunnel will retry as needed.)')

## 5 · Open an ngrok tunnel

In [ ]:
from pyngrok import ngrok, conf

conf.get_default().auth_token = os.environ['NGROK_AUTHTOKEN']
tunnel = ngrok.connect(PORT, 'http')
public_url = tunnel.public_url.replace('http://', 'https://')
print('\n' + '='*60)
print('PRAMAAN — sovereign LLM endpoint is LIVE')
print('='*60)
print(f'PRAMAAN_LLM_BASE_URL={public_url}/v1')
print(f'PRAMAAN_LLM_API_KEY=anything-non-empty')
print(f'PRAMAAN_LLM_EXTRACTOR_MODEL={MODEL_ID}')
print(f'PRAMAAN_LLM_SKEPTIC_MODEL={MODEL_ID}')
print('='*60)
print('Paste those into your local .env, then `make backend` in your repo.')

## 6 · Smoke test (optional)

In [ ]:
import time, requests, json

for _ in range(20):
    try:
        r = requests.get(f'http://localhost:{PORT}/v1/models', timeout=5)
        if r.ok:
            print(json.dumps(r.json(), indent=2))
            break
    except Exception:
        pass
    print('vLLM not ready yet, waiting 15s...')
    time.sleep(15)
else:
    print('vLLM did not become ready in time. Check the streaming logs above.')

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=f'http://localhost:{PORT}/v1', api_key='unused')
rsp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{'role': 'user', 'content': 'In one sentence: what is the role of an audit ledger in government procurement?'}],
    temperature=0,
    seed=42,
)
print(rsp.choices[0].message.content)

## 7 · Keep the cell running

Colab will reclaim idle runtimes. Keep this notebook tab open while you demo PRAMAAN locally. When you're done, run the next cell to clean up.

In [ ]:
# ngrok.disconnect(tunnel.public_url)
# proc.terminate()
# print('Stopped.')